## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Bajar datasets si hace falta

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

## Para correr con extensión de Colab en Visual Studio Code

In [3]:
import plotly.io as pio

# Set the renderer explicitly for VS Code Notebooks
pio.renderers.default = "vscode"  # alternative: 'notebook_mimetype'


# 1  Modelo AutoARIMA

## 1.1 Init Experimento

In [4]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 68.8 MB/s eta 0:00:00:00:0100:01


In [5]:
import os as os
import numpy as np
import polars as pl
import plotly.express as px
from pmdarima import auto_arima

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [6]:
# defino los parametros
PARAM = {'experimento':'AAideal01',
  'semilla_primigenia':102191
}

In [7]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AAideal01


## 1.2 Init AutoARIMA

In [8]:
# cargo el dataset del sell-in
tb_ventas = pl.read_csv('https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/tb_ventas_ideal.csv')

In [9]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/tb_realidad_ideal.csv')


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

4712
4712


In [10]:

producto= 20001
tbl = tb_ventas.filter(pl.col("product_id") == producto)

gr = px.line(
    y=tbl.select("tn").to_series(),
    title="product_id " + str(producto)
)

gr.show()

In [11]:
producto= 20002
tbl = tb_ventas.filter(pl.col("product_id") == producto)

gr = px.line(
    y=tbl.select("tn").to_series(),
    title="product_id " + str(producto)
)

gr.show()

Opcion de empiojar el dataset, agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [12]:
empiojar=False
empiojar_ruido=0.2

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


In [13]:
tb_ventas

product_id,periodo,tn,tn_real
i64,i64,f64,f64
20001,201601,5.835674,8.792946
20001,201602,3.832139,8.792946
20001,201603,3.908288,8.792946
20001,201604,6.001083,8.792946
20001,201605,8.19339,8.792946
…,…,…,…
20100,201908,553.136337,502.130103
20100,201909,631.076258,502.130103
20100,201910,516.864043,502.130103


In [14]:
# donde voy a guardar el resultado final
arch_tb_final = 'tb_final.csv'

try:
    # Attempt to load the existing file
    tb_final = pl.read_csv(arch_tb_final)
except FileNotFoundError:
    # donde voy a guardar el resultado final
    tb_final = tb_apredecir.clone()
    tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])


# cuento cuantos registros NO puedieron calcularse
# en mi corrida esto da  209

tb_final["tn"].is_null().sum()


100

In [15]:
correrdeCero=True

if correrdeCero:
  tb_final = tb_apredecir.clone()
  tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])

In [16]:
display(tb_final)

product_id,tn_real,tn
i64,f64,f64
20001,8.792946,null
20002,133.251123,null
20003,24.126122,null
20004,9.768757,null
20005,29.861519,null
…,…,…
20096,74.42495,null
20097,98.042823,null
20098,19.069984,null


## 1.3 Primera pasada AutoARIMA

Tengo en cuenta la estacionalidad de 12 periodos
<br> muchos van a quedar SIN  calcular, porque no tienen suficiente historia


In [17]:
# solo los productos que faltan

productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()


# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal=True,
      m=12, # estacinalidad cada 12
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10 # cantidad de iteraciones de busqueda AUTO
    )

    # predigo el periodo+1 y periodo+2
    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]  # el segundo elemento del vector

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0 )
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20001 20002 20003 20004 20005 20006 20007 20008 20009 20010 20011 20012 20013 20014 20015 20016 20017 20018 20019 20020 20021 20022 20023 20024 20025 20026 20027 20028 20029 20030 20031 20032 20033 20034 20035 20036 20037 20038 20039 20040 20041 20042 20043 20044 20045 20046 20047 20048 20049 20050 20051 20052 20053 20054 20055 20056 20057 20058 20059 20060 20061 20062 20063 20064 20065 20066 20067 20068 20069 20070 20071 20072 20073 20074 20075 20076 20077 20078 20079 20080 20081 20082 20083 20084 20085 20086 20087 20088 20089 20090 20091 20092 20093 20094 20095 20096 20097 20098 20099 20100 

In [18]:
# cuento cuantos registros NO puedieron calcularse

tb_final["tn"].is_null().sum()

0

In [19]:
# grabo la historia

tb_final.write_csv(arch_tb_final)

In [20]:
tb_final

product_id,tn_real,tn
i64,f64,f64
20001,8.792946,8.793218
20002,133.251123,133.632663
20003,24.126122,24.131888
20004,9.768757,9.768717
20005,29.861519,29.861515
…,…,…
20096,74.42495,74.431679
20097,98.042823,98.07484
20098,19.069984,19.069626


In [21]:
deno = (tb_final["tn_real"]).sum()
num = (tb_final["tn_real"] - tb_final["tn"]).abs().sum()

total_error_rate = num / deno
display( total_error_rate )

0.025450359501436943